# First EUV experiment on Colab

This notebook is an experiment control panel. It calls the Python code in `src/`; it does not contain a separate optimizer.

**Flow:** notebook configuration → `run_optimization.py` → `example_bayesopt.py` → `srcmdl.py` → saved results → notebook display.

A **free CPU runtime** is enough for the baseline GP. To test GPU access too, select a GPU runtime and run the optional GPU check below. The existing NumPy/scikit-learn GP models still execute on CPU.

**Scope:** this is a small execution test of the supplied GP optimizer. Its yield term is still a soft penalty, not a hard 99.5% constraint. A successful run does not mean that requirement has been implemented or validated.

## Before running cells

**VS Code:** install Google's official **Colab** extension (and Jupyter if prompted). Open this notebook, choose **Select Kernel → Colab → Auto Connect**, and sign in. For a GPU access test, choose **New Colab Server** instead of Auto Connect and select an available free GPU (for example T4, if offered). For the CPU baseline alone, Auto Connect is sufficient. Then right-click the local **`src` folder → Upload to Colab**. Uploading the folder should create `/content/src`.

**Colab website alternative:** upload this `.ipynb` through File → Upload notebook. For a GPU test, choose **Runtime → Change runtime type → a free GPU option → Save**, then Connect. Upload the five Python files inside `src/` using Colab's Files pane. The setup cell accepts these files directly in `/content` too.

You do not need the historical CSV logs, MATLAB files, PDFs, or local `.venv` for this test. Re-upload source files after changing them locally; the cloud copy is separate.

Run cells in order with their ▶ buttons. The same notebook also works with a local Python kernel if the required packages and IPython are installed.

In [ ]:
from pathlib import Path
import importlib.util
import platform
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
print("Python:", sys.version.split()[0])
print("Machine:", platform.platform())
print("Runtime:", "Colab" if IN_COLAB else "local / other Jupyter")
print("Working directory:", Path.cwd())
# In Colab, paths like /Users/josef/Desktop/ASML are not available.

## Optional: verify GPU access before running the GP

On a GPU runtime, `nvidia-smi` should show the assigned NVIDIA GPU. If PyTorch is installed and CUDA is available, the next cell also runs a tiny matrix multiplication on the GPU and prints its device. This verifies actual GPU computation independently of the CPU-based GP.

A GPU name plus `cuda:0` and `finite result: True` confirms that check passed. If no GPU is available, you can still run the entire baseline experiment on CPU. Free GPU access depends on availability and account usage. Selecting a GPU runtime alone does not make scikit-learn use it. Remove/disconnect the GPU server after saving your results.

In [ ]:
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=True)
else:
    print("No NVIDIA GPU detected. Select a GPU Colab runtime to test GPU access.")

if importlib.util.find_spec("torch") is not None:
    import torch
    if torch.cuda.is_available():
        with torch.no_grad():
            a = torch.randn((512, 512), device="cuda")
            product = a @ a.T
            finite = bool(torch.isfinite(product).all().item())
        torch.cuda.synchronize()
        print("GPU:", torch.cuda.get_device_name(0))
        print("Matrix multiplication device:", product.device, "| finite result:", finite)
        assert product.is_cuda and finite
        del a, product
    else:
        print("PyTorch is installed, but CUDA is unavailable on this runtime.")
else:
    print("PyTorch is not installed; nvidia-smi above checks GPU visibility only.")
print("The baseline Gaussian process below uses CPU, regardless of this GPU check.")

## 1. Find the uploaded code and prepare dependencies

The local `requirements.txt` records the original environment's exact versions. For this first cloud compatibility test, use the six required scientific libraries available to the Colab runtime; the runner records their actual versions. This does not guarantee numerical agreement across different package versions or platforms.

If the next cell cannot find the code, upload `src/` and run it again. Installing packages happens on the selected runtime, not your Mac's `.venv`.

In [ ]:
import subprocess

required_files = ["run_optimization.py", "example_bayesopt.py", "srcmdl.py", "dy_models.py"]
candidates = [Path("/content"), Path("/content/ASML"), Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = None
SOURCE_DIR = None
for candidate in candidates:
    for source in (candidate / "src", candidate):
        if all((source / name).is_file() for name in required_files):
            PROJECT_ROOT = candidate
            SOURCE_DIR = source
            break
    if SOURCE_DIR is not None:
        break
if SOURCE_DIR is None:
    raise FileNotFoundError(
        "Upload the src folder to Colab first (VS Code: right-click src > Upload to Colab). "
        "In browser Colab you can upload its Python files directly into /content."
    )

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "numpy", "scipy", "matplotlib", "scikit-learn", "statsmodels", "pandas"], check=True)
else:
    print("Using the local kernel's installed dependencies; no packages were installed.")
print("Source files:", SOURCE_DIR)
print("Results will be saved under:", PROJECT_ROOT / "results")

## 2. Choose the small experiment

Keep these defaults for the first test. The optimizer needs at least 27 initial measurements before it can select new settings. With 40 total measurements, up to 13 will be optimizer-selected. The CSV records how many actually were selected.

`DESIGN_SEED` controls the initial designs/candidate sampling. `SOURCE_SEED` selects the simulated source. Keep both fixed when comparing repeat runs. Yield model `1` is logit GP regression.

In [ ]:
N_MEASUREMENTS = 40
N_CANDIDATES = 1000
DESIGN_SEED = 0
SOURCE_SEED = 0
DY_MODEL = 1
COST_OPTION = 2  # Existing logit yield penalty; not a hard yield constraint.

## 3. Run the existing experiment script

This is the same script you run from the terminal locally. Each run gets a new output folder. Detailed measurement progress is written to `run.log`; the cell prints the result location when finished.

In [ ]:
from datetime import datetime, timezone
import os

run_name = "colab_test_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = PROJECT_ROOT / "results" / run_name
runtime_env = os.environ.copy()
runtime_env.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "asml-colab-matplotlib"))
command = [sys.executable, str(SOURCE_DIR / "run_optimization.py"),
           "--n", str(N_MEASUREMENTS), "--n-test", str(N_CANDIDATES),
           "--seed", str(DESIGN_SEED), "--source-seed", str(SOURCE_SEED),
           "--dy-model", str(DY_MODEL), "--cost-option", str(COST_OPTION),
           "--output", str(RUN_DIR)]
print("Running", N_MEASUREMENTS, "measurements...", flush=True)
subprocess.run(command, cwd=PROJECT_ROOT, env=runtime_env, check=True)
print("Completed:", RUN_DIR)

## 4. Inspect the plot and candidate recommendation

Each measurement is one tested set of nine settings. The plot's dotted line marks the first optimizer-selected measurement. The table below selects the highest measured EUV among points meeting the measured shape limits and **underlying simulated yield ≥99.5%**.

Underlying yield is available only in the simulator and is used here for diagnosis, not to train the optimizer or choose its next measurement. In a physical experiment, repeat measurements and a statistical acceptance rule would be needed. If no point qualifies, this notebook reports that rather than recommending an unqualified point.

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

summary = json.loads((RUN_DIR / "summary.json").read_text())
measurements = pd.read_csv(RUN_DIR / "measurements.csv")
display(Image(filename=str(RUN_DIR / "overview.png"), width=1000))
print("Optimizer-selected measurements:", summary["optimizer_selected_measurements"])
print("Shape + underlying yield feasible:", summary["shape_and_actual_yield_feasible_measurements"])
qualified = measurements.loc[measurements["shape_and_actual_yield_feasible"]]
if qualified.empty:
    print("No tested setting met the shape and underlying 99.5% yield requirements.")
else:
    winner = qualified.loc[[qualified["euv_mJ"].idxmax()]]
    display(winner[["measurement", "x", "y", "z", "e", "d", "p", "k", "b",
                    "dose_target_mJ", "euv_mJ", "observed_yield", "actual_yield_simulation_only"]].T)
print("Recorded package versions:", summary["packages"])

## 5. Keep the results after the Colab session ends

Your notebook stays saved locally in VS Code, but files created on the Colab runtime do not automatically appear in your Mac's `results/` folder. Colab runtimes are temporary.

This cell packages the results as a ZIP. To keep a persistent cloud copy, set **`SAVE_TO_DRIVE = True`** and run the cell; follow Google's authorization prompt. It copies only this run's ZIP to `My Drive/ASML/results/`. You can then download it from Drive to the local repository's `results/` folder. Google's official VS Code extension supports Drive mounting.

If using Colab in a browser, `DOWNLOAD_IN_BROWSER = True` also triggers a direct download. Keep that switch false inside VS Code. Save the notebook with Cmd+S as well.

In [ ]:
import shutil

SAVE_TO_DRIVE = False  # Set True in Colab to persist the results ZIP.
DOWNLOAD_IN_BROWSER = False  # Browser Colab only.
archive = Path(shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR))
print("Results archive:", archive)
if SAVE_TO_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Drive mounting is for a Colab runtime; local results are already on disk.")
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/ASML/results")
    destination.mkdir(parents=True, exist_ok=True)
    saved = shutil.copy2(archive, destination / archive.name)
    print("Persistent Drive copy:", saved)
if DOWNLOAD_IN_BROWSER:
    if not IN_COLAB:
        raise RuntimeError("Use the local archive path above outside Colab.")
    from google.colab import files
    files.download(str(archive))
if IN_COLAB and not SAVE_TO_DRIVE and not DOWNLOAD_IN_BROWSER:
    print("The ZIP is still on the temporary runtime. Enable a save option before disconnecting.")

## What counts as a successful first test?

- The runtime information confirms Colab execution.
- The experiment finishes and the plot appears.
- `optimizer_selected_measurements` is greater than zero.
- The results ZIP is copied to Drive or downloaded.

This demonstrates the notebook/cloud workflow. It does not establish model quality, MATLAB agreement, or a hard yield constraint. The next algorithm change is to express maximizing EUV subject to the yield threshold explicitly.

### Official setup references

- [Google Colab VS Code extension](https://marketplace.visualstudio.com/items?itemName=google.colab)
- [Google's VS Code guide: connections, file upload, and Drive mounting](https://github.com/googlecolab/colab-vscode/wiki/User-Guide)
- [Colab FAQ: free resources and temporary runtimes](https://research.google.com/colaboratory/faq.html)